# 01 - Zero-shot OWLv2 base (ensemble)
**Cantaloupe flowers - zero-shot pseudo-labels (no ground truth)**

Runs OWLv2 base (ensemble) (google/owlv2-base-patch16-ensemble) without training. Prompts and the score threshold are chosen on the validation split with a label-free score, then every image gets pseudo-label boxes. OWLv2 (Minderer et al., NeurIPS 2023) can also search with image exemplars cut from the photos themselves, so the result does not rest on prompt wording alone.

No training happens here, so there are no loss or metric curves. The dataset has no labels, so no accuracy is reported: the numbers measure whether the boxes stay the same when the photo is mirrored, and across the grayscale / flipped Roboflow copies.

Dataset: https://www.kaggle.com/datasets/samirhossain2001/cantaloupe-flower-datasets

**Attach as inputs:** the dataset + the output of notebook 00

## 1 - Setup
Pinned `transformers` so the model classes match the version this notebook was tested with.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'transformers==5.17.0'], check=True)

In [ ]:
import os
import gc
import re
import json
import glob
import time
import random
import hashlib
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

NB_START = time.time()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cv2", cv2.__version__,
      "| device", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| cpu cores", os.cpu_count())

## 2 - Configuration

In [ ]:
@dataclass
class CFG:
    INPUT_ROOTS: list = field(default_factory=lambda: os.environ.get("INPUT_ROOTS", '/kaggle/input').split(os.pathsep))
    OUT_DIR: str = '/kaggle/working' if os.path.isdir('/kaggle/working') else "outputs"
    DATA_DIR: str = '/kaggle/input/datasets/samirhossain2001/cantaloupe-flower-datasets/Cantaloupe flower datasets'  # dataset folder on Kaggle, checked first; otherwise the inputs are searched
    MODEL_NAME: str = 'OWLv2 base (ensemble)'
    TAG: str = 'owlv2'
    CHECKPOINT: str = 'google/owlv2-base-patch16-ensemble'
    PROMPTS: dict = field(default_factory=lambda: {'flower': ['flower'], 'yellow_flower': ['yellow flower'], 'melon_flower': ['melon flower'], 'flower_vs_lookalikes': ['yellow flower', 'flower bud', 'small green fruit', 'leaf'], 'five_petal_flower': ['open yellow flower with five petals']})  # name -> phrases; phrase 0 is the flower, the others absorb look-alikes
    THRESHOLDS: list = field(default_factory=lambda: [0.08, 0.1, 0.12, 0.15, 0.2, 0.25, 0.3, 0.4])  # score thresholds tried on validation
    MIN_SCORE: float = 0.05  # candidate boxes below this are dropped
    NMS_IOU: float = 0.5
    MAX_DET: int = 30
    MAX_BOX_AREA: float = 0.5  # a flower never covers half the frame in the montages; larger boxes are dropped
    CONTAIN_FRAC: float = 0.9  # a box that covers this share of a higher-scoring box is dropped (oversized / nested boxes)
    MATCH_IOU: float = 0.5  # IoU for two boxes to count as the same flower
    MAX_MEAN_BOXES: float = 6.0  # the montages show at most about 6 flowers per photo; settings above this mean are not chosen
    BATCH_SIZE: int = 4
    N_BOOT: int = 1000
    N_EXEMPLARS: int = 4  # train boxes whose embeddings form the image query
    EXEMPLAR_THRESHOLDS: list = field(default_factory=lambda: [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9])
    SEED: int = 42


cfg = CFG()
SMOKE = False
cfg

## 3 - Shared helpers
Image IO, flips / rotations, box matching, bootstrap.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def find_artifact(fname, required=True):
    for root in cfg.INPUT_ROOTS:
        hits = sorted(glob.glob(os.path.join(root, "**", fname), recursive=True))
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(f"{fname} not found: attach the output of the notebook that writes it as an input.")
    return None


def find_data_root(rel_path):
    if os.path.exists(os.path.join(cfg.DATA_DIR, rel_path)):
        return cfg.DATA_DIR
    for root in cfg.INPUT_ROOTS:
        for dirpath, _, _ in os.walk(root):
            if os.path.exists(os.path.join(dirpath, rel_path)):
                return dirpath
    raise FileNotFoundError(f"{rel_path} not found: attach the dataset as an input.")


def read_rgb(rel_path):
    a = cv2.imread(os.path.join(DATA_ROOT, rel_path), cv2.IMREAD_COLOR)
    assert a is not None, f"unreadable image: {rel_path}"
    return cv2.cvtColor(a, cv2.COLOR_BGR2RGB)


def dihedral(a, k):
    """k = 0..3: rotate k x 90 degrees counter-clockwise; k = 4..7: the same, then mirror left-right."""
    a = np.rot90(a, k % 4)
    return np.ascontiguousarray(a[:, ::-1] if k >= 4 else a)


def inv_k(k):
    """Inverse of dihedral(., k): rotations undo with the opposite turn, mirrored ones are their own inverse."""
    return k if k >= 4 else (4 - k) % 4


def dihedral_points(xy, k, w, h):
    xy = np.asarray(xy, dtype=np.float64).reshape(-1, 2).copy()
    for _ in range(k % 4):
        xy = np.stack([xy[:, 1], w - xy[:, 0]], axis=1)
        w, h = h, w
    if k >= 4:
        xy[:, 0] = w - xy[:, 0]
    return xy


def dihedral_boxes(boxes, k, w, h):
    boxes = np.asarray(boxes, dtype=np.float64).reshape(-1, 4)
    if len(boxes) == 0:
        return boxes
    p1, p2 = dihedral_points(boxes[:, :2], k, w, h), dihedral_points(boxes[:, 2:], k, w, h)
    return np.concatenate([np.minimum(p1, p2), np.maximum(p1, p2)], axis=1)


def box_area(b):
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    a, b = np.asarray(a, dtype=np.float64).reshape(-1, 4), np.asarray(b, dtype=np.float64).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(-1)
    return inter / np.maximum(box_area(a)[:, None] + box_area(b)[None, :] - inter, 1e-9)


def match_f1_from_iou(iou, n_a, n_b, thr):
    """F1 of a one-to-one matching at IoU >= thr; nan when both sides are empty."""
    if n_a + n_b == 0:
        return float("nan")
    if n_a == 0 or n_b == 0:
        return 0.0
    r, c = linear_sum_assignment(-iou)
    return 2 * int((iou[r, c] >= thr).sum()) / (n_a + n_b)


def match_f1(a, b, thr):
    return match_f1_from_iou(box_iou(a, b), len(a), len(b), thr)


def boot_ci(values, n_boot, seed=0):
    v = np.asarray([x for x in values if not np.isnan(x)], dtype=np.float64)
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    means = np.random.default_rng(seed).choice(v, (n_boot, len(v))).mean(1)
    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def hex_rgb(h):
    return tuple(int(h[i:i + 2], 16) for i in (1, 3, 5))


def tag_rect(img, text, x, y, scale):
    """Rectangle of a text tag whose bottom-left corner is (x, y), kept inside the image."""
    (tw, th), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    x = int(min(max(x, 0), img.shape[1] - tw - 6))
    y = int(min(max(y, th + base + 4), img.shape[0]))
    return x, y - th - base - 4, x + tw + 6, y, base


def put_tag(img, text, x, y, rgb, scale):
    x1, y1, x2, y2, base = tag_rect(img, text, x, y, scale)
    cv2.rectangle(img, (x1, y1), (x2, y2), rgb, -1)
    cv2.putText(img, text, (x1 + 3, y2 - base - 1), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2, cv2.LINE_AA)
    return x1, y1, x2, y2


def draw_boxes(img, boxes, scores=None, color="#2a78d6", numbers=False, thick=None):
    """Thick box with a dark outline, so it shows on leaves and on bright sky; label on a filled tag at the first
    corner of the box where it does not cover an earlier tag."""
    out = img.copy()
    rgb = hex_rgb(color)
    side = max(out.shape[:2])
    t = thick or max(3, round(side / 160))
    scale = max(0.6, side / 900)
    placed = []
    for i, b in enumerate(np.asarray(boxes).reshape(-1, 4).astype(int)):
        p1, p2 = (int(b[0]), int(b[1])), (int(b[2]), int(b[3]))
        cv2.rectangle(out, p1, p2, (20, 20, 20), t + 3, cv2.LINE_AA)
        cv2.rectangle(out, p1, p2, rgb, t, cv2.LINE_AA)
        label = str(i) if numbers else (f"{scores[i]:.2f}" if scores is not None else "")
        if not label:
            continue
        (tw, th), base = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
        h = th + base + 4
        o = (t + 3) // 2
        corners = [(p1[0] - o, p1[1] - o), (p1[0] - o, p2[1] + o + h), (p2[0] - tw - 6 + o, p1[1] - o),
                   (p2[0] - tw - 6 + o, p2[1] + o + h), (p1[0] + o, p1[1] + o + h)]
        free = [c for c in corners if not any(
            r[0] < q[2] and q[0] < r[2] and r[1] < q[3] and q[1] < r[3] for r in [tag_rect(out, label, *c, scale)[:4]] for q in placed)]
        placed.append(put_tag(out, label, *(free[0] if free else corners[0]), rgb, scale))
    return out


def finish(fig, title):
    """Title above the panels (also for tall figures), then show."""
    h = fig.get_figheight()
    fig.suptitle(title, y=1 - 0.12 / h, va="top")
    plt.tight_layout(rect=(0, 0, 1, 1 - 0.5 / h))
    plt.show()


def save_json(obj, name):
    path = os.path.join(cfg.OUT_DIR, name)
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)
    return path


os.makedirs(cfg.OUT_DIR, exist_ok=True)
seed_everything(cfg.SEED)

probe = np.zeros((40, 60), np.uint8)
probe[5:12, 30:50] = 1
for k in range(8):
    ys, xs = np.nonzero(dihedral(probe, k))
    assert np.allclose(dihedral_boxes([[30, 5, 50, 12]], k, 60, 40)[0], [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]), k
    assert np.array_equal(dihedral(dihedral(probe, k), inv_k(k)), probe), k
print("box transform matches the image transform, and each inverse undoes it, for all 8 flips / rotations")

## 4 - Load the split from 00

In [ ]:
split_meta = json.load(open(find_artifact("split.json")))
ITEMS = pd.DataFrame([dict(it, split=s) for s, its in split_meta["items"].items() for it in its])
DATA_ROOT = find_data_root(ITEMS.img.iloc[0])
QUAL = split_meta["qual_sources"]
if SMOKE:
    pick = []
    for s in ("train", "validation", "test"):
        part = ITEMS[ITEMS.split == s]
        pick += list(part.groupby("source").size().sort_values(ascending=False, kind="stable").index[:3])
    pick += [q for q in QUAL if q in set(ITEMS.source)][:2]
    ITEMS = ITEMS[ITEMS.source.isin(pick)].reset_index(drop=True)
    QUAL = [q for q in QUAL if q in set(ITEMS.source)][:2]
BY_ID = ITEMS.set_index("id", drop=False)
print(f"split {split_meta['split_id']} | images per split:", ITEMS.split.value_counts().to_dict(),
      "| multi-copy sources:", {s: int((g.groupby("source").size() > 1).sum()) for s, g in ITEMS.groupby("split")})

## 5 - Model
The adapter returns every candidate box with the phrase it matches best. Only boxes whose best phrase is the flower phrase are kept.

In [ ]:
def autocast():
    return torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda")


def cxcywh_to_xyxy(b):
    return torch.cat([b[..., :2] - b[..., 2:] / 2, b[..., :2] + b[..., 2:] / 2], dim=-1)


def unpack(cls_probs, boxes_px):
    """cls_probs (B, Q, C) per phrase; label = best phrase, score = its probability. Phrase 0 is the flower."""
    res = []
    for p, b in zip(cls_probs, boxes_px):
        scores, labels = p.max(-1)
        res.append({"boxes": b.cpu().numpy(), "scores": scores.cpu().numpy(), "labels": labels.cpu().numpy(),
                    "q": np.arange(len(b))})
    return res

In [ ]:
from transformers import Owlv2Processor, Owlv2ForObjectDetection

processor = Owlv2Processor.from_pretrained(cfg.CHECKPOINT)
model = Owlv2ForObjectDetection.from_pretrained(cfg.CHECKPOINT).to(device).eval()


def scale_square(boxes, images):
    """OWLv2 pads each image to a square, so boxes are relative to the longer side."""
    s = torch.tensor([max(im.shape[:2]) for im in images], dtype=boxes.dtype, device=boxes.device)
    return boxes * s[:, None, None]


@torch.no_grad()
def detect(images, prompt):
    if prompt == "exemplar":
        return detect_exemplar(images)
    inputs = processor(text=[list(prompt)] * len(images), images=images, return_tensors="pt").to(device)
    with autocast():
        out = model(**inputs)
    return unpack(out.logits.float().sigmoid(), scale_square(cxcywh_to_xyxy(out.pred_boxes.float()), images))


@torch.no_grad()
def class_embeds_at(images, prompt, q_idx):
    inputs = processor(text=[list(prompt)] * len(images), images=images, return_tensors="pt").to(device)
    out = model(**inputs)
    return torch.stack([out.class_embeds[k, q] for k, q in enumerate(q_idx)]).float()


@torch.no_grad()
def detect_exemplar(images):
    """Score = cosine similarity between each patch's class embedding and the closest exemplar (the sigmoid saturates)."""
    inputs = processor(images=images, return_tensors="pt").to(device)
    with autocast():
        feature_map = model.image_embedder(pixel_values=inputs["pixel_values"])[0]
        b, ph, pw, d = feature_map.shape
        feats = feature_map.reshape(b, ph * pw, d)
        q = EXEMPLAR_EMBEDS[None].expand(b, -1, -1).to(feats.dtype)
        _, class_embeds = model.class_predictor(feats, q, torch.ones(q.shape[:2], dtype=torch.bool, device=device))
        boxes = model.box_predictor(feats, feature_map)
    cos = torch.nn.functional.normalize(class_embeds.float(), dim=-1) @ EXEMPLAR_EMBEDS.T.float()
    return unpack(cos.max(-1, keepdim=True)[0], scale_square(cxcywh_to_xyxy(boxes.float()), images))


print(f"{cfg.MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters")

## 6 - Label-free scoring
Main check, mirror consistency: the colour reference of each photo is also run mirrored left-right, the boxes are mapped back and matched at IoU >= `MATCH_IOU`; F1 per photo. Every train photo has exactly one colour copy, so the Roboflow copies only test colour-vs-gray robustness; they are reported as a second check. Selection score = mirror F1 x share of photos with at least one box, among settings with at most `MAX_MEAN_BOXES` boxes per photo (many boxes match by chance). After NMS, a box that almost fully contains a higher-scoring box is dropped (`CONTAIN_FRAC`).

In [ ]:
def postprocess(d, w, h):
    b = np.asarray(d["boxes"], dtype=np.float64).reshape(-1, 4)
    b = np.stack([b[:, 0].clip(0, w), b[:, 1].clip(0, h), b[:, 2].clip(0, w), b[:, 3].clip(0, h)], axis=1)
    s, lab, q = np.asarray(d["scores"]), np.asarray(d["labels"]), np.asarray(d["q"])
    frac = box_area(b) / (w * h)
    keep = (lab == 0) & (s >= cfg.MIN_SCORE) & (frac > 0) & (frac <= cfg.MAX_BOX_AREA)
    b, s, q = b[keep], s[keep], q[keep]
    k = torchvision.ops.nms(torch.as_tensor(b, dtype=torch.float32), torch.as_tensor(s, dtype=torch.float32), cfg.NMS_IOU).numpy()
    b, s, q = b[k], s[k], q[k]
    keep = []
    for j in range(len(b)):
        if keep:
            lt = np.maximum(b[keep, :2], b[j, :2])
            rb = np.minimum(b[keep, 2:], b[j, 2:])
            inter = np.clip(rb - lt, 0, None).prod(-1)
            if (inter / np.maximum(box_area(b[keep]), 1e-9) >= cfg.CONTAIN_FRAC).any():
                continue
        keep.append(j)
    keep = np.array(keep[:cfg.MAX_DET], dtype=np.int64)
    return {"boxes": b[keep].astype(np.float32), "scores": s[keep].astype(np.float32), "q": q[keep].astype(np.int64)}


def run_detector(ids, prompt, desc, mirror=False):
    """mirror=True runs on the left-right mirrored image and maps the boxes back to the original frame."""
    out = {}
    for s in tqdm(range(0, len(ids), cfg.BATCH_SIZE), desc=desc, leave=False):
        chunk = ids[s:s + cfg.BATCH_SIZE]
        imgs = [read_rgb(BY_ID.img[i]) for i in chunk]
        if mirror:
            imgs = [dihedral(im, 4) for im in imgs]
        for i, img, d in zip(chunk, imgs, detect(imgs, prompt)):
            h, w = img.shape[:2]
            r = postprocess(d, w, h)
            if mirror:
                r["boxes"] = dihedral_boxes(r["boxes"], 4, w, h).astype(np.float32)
            out[i] = r
    return out


def at_threshold(raw, thr):
    return {i: d["boxes"][d["scores"] >= thr] for i, d in raw.items()}


def pair_kind(a, b):
    return "gray-gray" if a.is_gray and b.is_gray else ("colour-colour" if not a.is_gray and not b.is_gray else "colour-gray")


def copy_pairs(ids, thr_iou, boxes):
    """One row per pair of aligned Roboflow copies of the same photo; boxes are moved into the reference frame first."""
    part = BY_ID.loc[[i for i in ids if i in boxes]]
    rows = []
    for src, g in part[part.aligned].groupby("source"):
        its = list(g.itertuples())
        ref = {it.id: dihedral_boxes(boxes[it.id], it.tf, it.width, it.height) for it in its}
        for x in range(len(its)):
            for y in range(x + 1, len(its)):
                a, b = its[x], its[y]
                rows.append({"source": src, "kind": pair_kind(a, b), "f1": match_f1(ref[a.id], ref[b.id], thr_iou)})
    return pd.DataFrame(rows, columns=["source", "kind", "f1"])


def summarise(ids, boxes, mirror_boxes):
    refs = [i for i in ids if BY_ID.is_ref[i]]
    mir = pd.Series([match_f1(boxes[i], mirror_boxes[i], cfg.MATCH_IOU) for i in refs], index=refs, dtype=float)
    pairs = copy_pairs(ids, cfg.MATCH_IOU, boxes)
    n = np.array([len(boxes[i]) for i in refs])
    mf = float(mir.mean()) if mir.notna().any() else float("nan")
    rate = float((n > 0).mean()) if len(n) else 0.0
    det = float(n.mean()) if len(n) else 0.0
    return {"mirror_f1": mf, "copy_f1": float(pairs.groupby("source").f1.mean().mean()) if pairs.f1.notna().any() else float("nan"),
            "share_ref_with_det": rate, "det_per_ref": det, "n_photos": int(mir.notna().sum()), "n_both_empty": int(mir.isna().sum()),
            "selection_score": (0.0 if np.isnan(mf) else mf) * rate, "plausible": bool(det <= cfg.MAX_MEAN_BOXES),
            "mirror_per_photo": mir, "pairs": pairs}


def sweep_rows(name, raw, raw_m, ids, thresholds):
    return [{"prompt": name, "threshold": thr, **{k: v for k, v in summarise(ids, at_threshold(raw, thr), at_threshold(raw_m, thr)).items()
                                                  if k not in ("pairs", "mirror_per_photo")}} for thr in thresholds]

In [ ]:
quick_prompt = cfg.PROMPTS.get("yellow_flower", next(iter(cfg.PROMPTS.values())))
quick_thr = sorted(cfg.THRESHOLDS)[len(cfg.THRESHOLDS) // 2]
refs = [BY_ID[(BY_ID.source == s) & BY_ID.is_ref].id.iloc[0] for s in QUAL[:4]]
raw_q = run_detector(refs, quick_prompt, "quick look")
fig, axes = plt.subplots(1, len(refs), figsize=(4.5 * len(refs), 4.8), squeeze=False)
for a, i in zip(axes[0], refs):
    d = raw_q[i]
    k = d["scores"] >= quick_thr
    a.imshow(draw_boxes(read_rgb(BY_ID.img[i]), d["boxes"][k], d["scores"][k]))
    a.set_title(f"{BY_ID.source[i]} | {int(k.sum())} boxes", fontsize=9)
    a.axis("off")
plt.suptitle(f"First look: prompt {quick_prompt}, score >= {quick_thr} (middle of the threshold grid)")
plt.tight_layout()
plt.show()

## 7 - Prompt and threshold sweep on validation
Each prompt runs once at `MIN_SCORE`; thresholds are applied afterwards.

In [ ]:
if SMOKE:
    cfg.PROMPTS = dict(list(cfg.PROMPTS.items())[:2])
    cfg.THRESHOLDS = cfg.THRESHOLDS[::3]
    if hasattr(cfg, "EXEMPLAR_THRESHOLDS"):
        cfg.EXEMPLAR_THRESHOLDS = cfg.EXEMPLAR_THRESHOLDS[::3]
val_ids = list(ITEMS[ITEMS.split == "validation"].id)
val_refs = [i for i in val_ids if BY_ID.is_ref[i]]
RAW_VAL, RAW_VAL_M, rows = {}, {}, []
for name, prompt in cfg.PROMPTS.items():
    RAW_VAL[name] = run_detector(val_ids, prompt, f"validation | {name}")
    RAW_VAL_M[name] = run_detector(val_refs, prompt, f"validation mirrored | {name}", mirror=True)
    rows += sweep_rows(name, RAW_VAL[name], RAW_VAL_M[name], val_ids, cfg.THRESHOLDS)
sweep = pd.DataFrame(rows)
display(sweep.sort_values("selection_score", ascending=False).head(10).round(3))

In [ ]:
def plot_sweep(sweep):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for k, (name, g) in enumerate(sweep.groupby("prompt", sort=False)):
        for a, col in zip(axes, ["mirror_f1", "det_per_ref", "selection_score"]):
            a.plot(g.threshold, g[col], marker="o", color=PALETTE[k % len(PALETTE)], label=name)
    axes[1].axhline(cfg.MAX_MEAN_BOXES, color="#8a8984", ls="--", label="plausible maximum")
    for a, t in zip(axes, ["Mirror consistency F1 (validation)", "Boxes per photo (validation)",
                           "Selection score = F1 x share of photos with a box"]):
        a.set_title(t, fontsize=10)
        a.set_xlabel("score threshold")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


plot_sweep(sweep)

## 8 - Image exemplars
OWLv2 can search with an image instead of text. The best text prompt finds the top flowers on colour train photos; their embeddings become the query. Scores are cosine similarities, because the sigmoid scores saturate near 1 for image queries.

In [ ]:
text_sweep = sweep[sweep.plausible] if sweep.plausible.any() else sweep
best_text = text_sweep.sort_values("selection_score", ascending=False).iloc[0]
train_refs = list(ITEMS[(ITEMS.split == "train") & ITEMS.is_ref & ~ITEMS.is_gray].id)
raw_tr = run_detector(train_refs, cfg.PROMPTS[best_text.prompt], "exemplar search (train)")
cand = sorted(((float(d["scores"][0]), i, int(d["q"][0])) for i, d in raw_tr.items() if len(d["scores"])), reverse=True)[:cfg.N_EXEMPLARS]
assert cand, "no exemplar found on the train photos"
EXEMPLAR_EMBEDS = torch.nn.functional.normalize(
    torch.cat([class_embeds_at([read_rgb(BY_ID.img[i])], cfg.PROMPTS[best_text.prompt], [q]) for _, i, q in cand]), dim=-1)
fig, axes = plt.subplots(1, len(cand), figsize=(3 * len(cand), 3), squeeze=False)
for a, (s, i, q) in zip(axes[0], cand):
    b = raw_tr[i]["boxes"][0].astype(int)
    a.imshow(read_rgb(BY_ID.img[i])[b[1]:b[3], b[0]:b[2]])
    a.set_title(f"text score {s:.2f}", fontsize=9)
    a.axis("off")
plt.suptitle(f"Exemplars: top {len(cand)} train boxes for '{best_text.prompt}' (their embeddings become the query)")
plt.tight_layout()
plt.show()

cfg.PROMPTS["exemplar"] = "exemplar"
RAW_VAL["exemplar"] = run_detector(val_ids, "exemplar", "validation | exemplar")
RAW_VAL_M["exemplar"] = run_detector(val_refs, "exemplar", "validation mirrored | exemplar", mirror=True)
sweep = pd.concat([sweep, pd.DataFrame(sweep_rows("exemplar", RAW_VAL["exemplar"], RAW_VAL_M["exemplar"], val_ids,
                                                  cfg.EXEMPLAR_THRESHOLDS))], ignore_index=True)
print("Exemplar scores are cosine similarities to the exemplar embeddings, so their thresholds differ from the text ones.")
display(sweep[sweep.prompt == "exemplar"].round(3))
plot_sweep(sweep)

## 9 - Chosen prompt and threshold

In [ ]:
eligible = sweep[sweep.plausible]
if eligible.empty:
    print(f"no setting stays under {cfg.MAX_MEAN_BOXES} boxes per photo: choosing among all of them")
    eligible = sweep
best = eligible.sort_values(["selection_score", "mirror_f1"], ascending=False).iloc[0]
PROMPT_NAME, THRESHOLD = best.prompt, float(best.threshold)
print(f"chosen on validation: prompt '{PROMPT_NAME}' = {cfg.PROMPTS[PROMPT_NAME]} | threshold {THRESHOLD} | "
      f"mirror F1 {best.mirror_f1:.3f} | photos with a box {best.share_ref_with_det:.0%} | boxes per photo {best.det_per_ref:.2f}")

## 10 - Pseudo-labels for every image
The model runs on every image. The saved labels come from the colour reference of each photo: its grayscale / flipped Roboflow copies get the same boxes through the transforms from 00, so all copies of a photo agree. Each copy's own detections are kept for the robustness check.

In [ ]:
t0 = time.time()
all_ids = list(ITEMS.id)
prompt = cfg.PROMPTS[PROMPT_NAME]
todo = [i for i in all_ids if i not in RAW_VAL[PROMPT_NAME]]
RAW = {**{i: RAW_VAL[PROMPT_NAME][i] for i in all_ids if i in RAW_VAL[PROMPT_NAME]}, **run_detector(todo, prompt, "all images")}
RUN_S = time.time() - t0
test_refs = list(ITEMS[(ITEMS.split == "test") & ITEMS.is_ref].id)
RAW_M = {**RAW_VAL_M[PROMPT_NAME], **run_detector(test_refs, prompt, "test mirrored", mirror=True)}
DET = {i: {"boxes": d["boxes"][d["scores"] >= THRESHOLD], "scores": d["scores"][d["scores"] >= THRESHOLD]} for i, d in RAW.items()}
MIRROR = at_threshold(RAW_M, THRESHOLD)

REF_OF = ITEMS[ITEMS.is_ref].set_index("source").id
FINAL, n_prop = {}, 0
for it in ITEMS.itertuples():
    r = REF_OF[it.source]
    if it.id == r or not it.aligned:
        FINAL[it.id] = DET[it.id]
    else:
        FINAL[it.id] = {"boxes": dihedral_boxes(DET[r]["boxes"], inv_k(it.tf), BY_ID.width[r], BY_ID.height[r]).astype(np.float32),
                        "scores": DET[r]["scores"]}
        n_prop += 1
print(f"{len(RAW)} images | {sum(len(d['boxes']) for d in DET.values())} boxes detected at threshold {THRESHOLD} | "
      f"{RUN_S / 60:.1f} min for {len(todo)} new images")
print(f"pseudo-labels: {n_prop} Roboflow copies take the boxes of their colour reference (flip undone) | "
      f"{sum(len(d['boxes']) for d in FINAL.values())} boxes in total")

## 11 - Label-free evaluation
Validation was used for the choice, so test is the number to quote. Roboflow copy pairs show how much the model depends on colour.

In [ ]:
boxes = {i: d["boxes"] for i, d in DET.items()}
RESULTS = {}
for s in ("validation", "test"):
    ids = list(ITEMS[ITEMS.split == s].id)
    r = summarise(ids, boxes, MIRROR)
    mean, lo, hi = boot_ci(r["mirror_per_photo"].values, cfg.N_BOOT, cfg.SEED)
    by_kind = {k: boot_ci(g.groupby("source").f1.mean().values, cfg.N_BOOT, cfg.SEED) for k, g in r["pairs"].groupby("kind")}
    RESULTS[s] = {**{k: v for k, v in r.items() if k not in ("pairs", "mirror_per_photo")}, "mirror_ci": [lo, hi],
                  "copy_by_kind": {k: {"f1": v[0], "ci": [v[1], v[2]]} for k, v in by_kind.items()},
                  "per_photo": {k: (None if np.isnan(v) else float(v)) for k, v in r["mirror_per_photo"].items()}}
    print(f"{s:10s} mirror consistency F1 {mean:.3f} [95% CI {lo:.3f}, {hi:.3f}] over {r['n_photos']} photos | "
          f"Roboflow copies, own detections: " + (", ".join(f"{k} {v[0]:.3f}" for k, v in by_kind.items()) or "none"))
print("These numbers measure stability (mirror, grayscale / flipped copies), not accuracy: there is no ground truth.")
print("The copy numbers use each copy's own detections; the saved pseudo-labels copy the reference boxes instead.")

fig, axes = plt.subplots(1, 4, figsize=(19, 3.8))
bars = {"mirror (colour)": [RESULTS["test"]["mirror_f1"], *RESULTS["test"]["mirror_ci"]],
        **{f"copies {k}": [v["f1"], *v["ci"]] for k, v in RESULTS["test"]["copy_by_kind"].items()}}
vals = np.array(list(bars.values()), dtype=np.float64)
axes[0].bar(list(bars), vals[:, 0], color=PALETTE[:len(bars)], yerr=np.abs(np.stack([vals[:, 0] - vals[:, 1], vals[:, 2] - vals[:, 0]])), capsize=4)
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", labelsize=8)
axes[0].set_title("Test consistency F1 (95% CI)", fontsize=10)
axes[1].hist([len(boxes[i]) for i in test_refs], bins=range(0, cfg.MAX_DET + 2), color="#2a78d6")
axes[1].set_title("Boxes per test photo", fontsize=10)
sc = np.concatenate([RAW[i]["scores"] for i in test_refs]) if test_refs else np.zeros(0)
axes[2].hist(sc, bins=40, color="#1baf7a")
axes[2].axvline(THRESHOLD, color="#eb6834", ls="--", label="threshold")
axes[2].set_title("Scores of all candidate boxes (test)", fontsize=10)
axes[2].legend()
side = np.concatenate([np.sqrt(box_area(boxes[i].astype(np.float64)) / (BY_ID.width[i] * BY_ID.height[i])) for i in test_refs if len(boxes[i])] or [np.zeros(0)])
axes[3].hist(side, bins=30, color="#eda100")
axes[3].set_title("Box size: sqrt(box area / image area)", fontsize=10)
plt.tight_layout()
plt.show()
per_date = pd.DataFrame({"date": [BY_ID.date[i] for i in test_refs], "boxes": [len(boxes[i]) for i in test_refs]})
per_date.groupby("date").boxes.agg(["mean", "size"]).round(2)

## 12 - Sample predictions

In [ ]:
def show_sources(srcs, title):
    srcs = [s for s in srcs if (ITEMS.source == s).any()]
    if not srcs:
        return
    fig, axes = plt.subplots(len(srcs), 3, figsize=(13, 4.3 * len(srcs)), squeeze=False)
    for r, s in enumerate(srcs):
        g = ITEMS[ITEMS.source == s].sort_values("is_ref", ascending=False)
        for c in range(3):
            axes[r, c].axis("off")
        for c, it in enumerate(g.head(3).itertuples()):
            img = dihedral(read_rgb(it.img), it.tf)
            b = dihedral_boxes(FINAL[it.id]["boxes"], it.tf, it.width, it.height)
            axes[r, c].imshow(draw_boxes(img, b, FINAL[it.id]["scores"]))
            axes[r, c].set_title(f"{s} | {'ref' if it.is_ref else 'copy'} {'gray' if it.is_gray else 'colour'} | {len(b)} boxes", fontsize=9)
    finish(fig, title)


def show_mirror(ids, title):
    if not ids:
        return
    fig, axes = plt.subplots(len(ids), 2, figsize=(9, 4.4 * len(ids)), squeeze=False)
    for r, i in enumerate(ids):
        img = read_rgb(BY_ID.img[i])
        axes[r, 0].imshow(draw_boxes(img, FINAL[i]["boxes"], FINAL[i]["scores"]))
        axes[r, 0].set_title(f"{BY_ID.source[i]} | original | {len(FINAL[i]['boxes'])} boxes", fontsize=9)
        axes[r, 1].imshow(draw_boxes(img, MIRROR[i], color="#eb6834"))
        axes[r, 1].set_title(f"mirrored pass, mapped back | {len(MIRROR[i])} boxes", fontsize=9)
        for a in axes[r]:
            a.axis("off")
    finish(fig, title)


show_sources(QUAL, "Fixed test photos from notebook 00: saved pseudo-labels (copies take the reference boxes; shown in the reference orientation)")
worst = pd.Series(RESULTS["test"]["per_photo"]).dropna().sort_values().index[:3].tolist()
show_mirror(worst, "Test photos where the original and the mirrored pass disagree most")

## 13 - Speed and saved outputs
Pseudo-labels as COCO json and YOLO txt (class 0 = flower), all candidate boxes, the mirrored-pass boxes for the comparison notebook, and metrics.

In [ ]:
one = [read_rgb(BY_ID.img[test_refs[0] if test_refs else all_ids[0]])]
for _ in range(2):
    detect(one, prompt)
if device.type == "cuda":
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
for _ in range(5):
    detect(one, prompt)
if device.type == "cuda":
    torch.cuda.synchronize()
latency_ms = (time.time() - t0) / 5 * 1000
params = sum(p.numel() for p in model.parameters())
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None


def coco_export(final):
    images, anns = [], []
    for k, it in enumerate(ITEMS.sort_values("id").itertuples(), 1):
        images.append({"id": k, "file_name": it.img, "item_id": it.id, "split": it.split, "width": int(it.width), "height": int(it.height)})
        for b, s in zip(final[it.id]["boxes"], final[it.id]["scores"]):
            x1, y1, x2, y2 = [float(v) for v in b]
            anns.append({"id": len(anns) + 1, "image_id": k, "category_id": 1, "bbox": [x1, y1, x2 - x1, y2 - y1],
                         "area": (x2 - x1) * (y2 - y1), "score": float(s), "iscrowd": 0})
    return {"info": {"tag": cfg.TAG, "model": cfg.MODEL_NAME, "prompt": PROMPT_NAME, "threshold": THRESHOLD,
                     "split_id": split_meta["split_id"], "propagated": True,
                     "note": "zero-shot pseudo-labels, not ground truth; Roboflow copies take the boxes of their colour reference"},
            "images": images, "annotations": anns, "categories": [{"id": 1, "name": "flower"}]}


def yolo_export(final, folder):
    for it in ITEMS.itertuples():
        d = os.path.join(cfg.OUT_DIR, folder, "labels", it.split)
        os.makedirs(d, exist_ok=True)
        with open(os.path.join(d, os.path.splitext(os.path.basename(it.img))[0] + ".txt"), "w") as f:
            for x1, y1, x2, y2 in final[it.id]["boxes"]:
                f.write(f"0 {(x1 + x2) / 2 / it.width:.6f} {(y1 + y2) / 2 / it.height:.6f} {(x2 - x1) / it.width:.6f} {(y2 - y1) / it.height:.6f}\n")
    with open(os.path.join(cfg.OUT_DIR, folder, "data.yaml"), "w") as f:
        f.write("# labels only; images are in the dataset. Pseudo-labels, not ground truth.\nnames:\n  0: flower\n")


save_json(coco_export(FINAL), f"pseudo_{cfg.TAG}.json")
yolo_export(FINAL, f"yolo_{cfg.TAG}")
save_json({i: {"boxes": d["boxes"].round(1).tolist(), "scores": d["scores"].round(4).tolist()} for i, d in RAW.items()}, f"raw_{cfg.TAG}.json")
save_json({i: np.asarray(b).round(1).tolist() for i, b in MIRROR.items()}, f"mirror_{cfg.TAG}.json")
metrics = {"tag": cfg.TAG, "model": cfg.MODEL_NAME, "checkpoint": cfg.CHECKPOINT, "split_id": split_meta["split_id"],
           "task": "zero_shot_detection", "main_metric": "mirror_consistency_f1", "higher_better": True, "bounded": True,
           "label_free": True, "prompt": PROMPT_NAME, "phrases": cfg.PROMPTS[PROMPT_NAME], "threshold": THRESHOLD,
           "min_score": cfg.MIN_SCORE, "match_iou": cfg.MATCH_IOU, "max_mean_boxes": cfg.MAX_MEAN_BOXES,
           "contain_frac": cfg.CONTAIN_FRAC, "copies_propagated": n_prop,
           "sweep": sweep.to_dict("records"),
           "validation": {k: v for k, v in RESULTS["validation"].items() if k != "per_photo"},
           "test": {k: v for k, v in RESULTS["test"].items() if k != "per_photo"},
           "params_m": params / 1e6, "latency_ms": latency_ms, "images_per_s": len(todo) / max(RUN_S, 1e-9) if todo else None,
           "peak_mem_gb": peak_gb, "smoke": SMOKE}
save_json(metrics, f"metrics_{cfg.TAG}.json")
save_json({s: RESULTS[s]["per_photo"] for s in RESULTS}, f"per_item_{cfg.TAG}.json")
print(f"params {params / 1e6:.1f} M | latency {latency_ms:.0f} ms / image (batch 1, {device.type}) | peak memory {peak_gb} GB")
print(f"saved pseudo_{cfg.TAG}.json, yolo_{cfg.TAG}/, raw_{cfg.TAG}.json, mirror_{cfg.TAG}.json, metrics_{cfg.TAG}.json, "
      f"per_item_{cfg.TAG}.json | total {(time.time() - NB_START) / 60:.1f} min")